In [1]:
import hashlib
import pandas as pd
import json


from pancancer_epigenetics.utils.paths import Paths
from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import project_relative_path

PROJECT_ROOT = Paths.root
RAW_DIR = Paths.root / "data" / "raw"
AUDIT_DIR = Paths.audit
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
REGISTRY_PATH = Paths.raw_data_registry


In [2]:
# Load the raw data registry
with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
    RAW_DATA_REGISTRY = json.load(f)

RAW_DATA_REGISTRY.keys()

dict_keys(['depmap', 'gdsc', 'ctrp', 'prism', 'tcga', 'lincs', 'cell_model_passports', 'chembl', 'drugbank'])

In [3]:
# =============================================================================
# Raw-file registry audit
#
# TCGA payloads are excluded because they are governed by cohort-specific
# GDC manifests and validated in dedicated download-validation notebooks.
# =============================================================================

MANIFEST_MANAGED_DATASETS = {"tcga"}

records = []
missing_metadata = []
excluded_file_counts = {}


# -----------------------------------------------------------------------------
# Inspect raw files
# -----------------------------------------------------------------------------

for file_path in RAW_DIR.rglob("*"):

    if not file_path.is_file():
        continue

    if file_path.name == ".gitkeep":
        continue

    raw_relative_path = file_path.relative_to(RAW_DIR)

    # The dataset is the first directory below data/raw,
    # not necessarily the file's immediate parent.
    dataset = raw_relative_path.parts[0]
    file_name = file_path.name

    # TCGA uses cohort manifests and dedicated validation notebooks.
    if dataset in MANIFEST_MANAGED_DATASETS:
        excluded_file_counts[dataset] = (
            excluded_file_counts.get(dataset, 0) + 1
        )
        continue

    dataset_metadata = RAW_DATA_REGISTRY.get(dataset)

    if dataset_metadata is None:
        missing_metadata.append(project_relative_path(file_path))
        continue

    file_metadata = dataset_metadata.get("files", {}).get(file_name)

    if file_metadata is None:
        missing_metadata.append(project_relative_path(file_path))
        continue

    records.append(
        {
            "relative_path": project_relative_path(file_path),
            "file_name": file_name,
            "dataset": dataset,
            "source_database": dataset_metadata.get("source_database"),
            "provider": dataset_metadata.get("provider"),
            "release": dataset_metadata.get("release"),
            "download_page_url": dataset_metadata.get("download_page_url"),
            "file_url": file_metadata.get("file_url"),
            "size_bytes": file_path.stat().st_size,
            "sha256": calculate_sha256(file_path),
        }
    )


# -----------------------------------------------------------------------------
# Report files excluded from the registry-level audit
# -----------------------------------------------------------------------------

if excluded_file_counts:
    print("Manifest-managed datasets excluded from this audit:")

    for dataset, file_count in sorted(excluded_file_counts.items()):
        print(f"  - {dataset}: {file_count:,} filesystem files")

    print()


# -----------------------------------------------------------------------------
# Report genuinely unregistered files
# -----------------------------------------------------------------------------

if missing_metadata:
    print("WARNING: Missing registry metadata for:")

    for item in missing_metadata:
        print(f"  - {item}")

    print()


# -----------------------------------------------------------------------------
# Build audit table
# -----------------------------------------------------------------------------

raw_file_audit = (
    pd.DataFrame(records)
    .sort_values(["dataset", "file_name"])
    .reset_index(drop=True)
)

audit_records = (
    raw_file_audit
    .astype(object)
    .where(pd.notna(raw_file_audit), None)
    .to_dict(orient="records")
)


# -----------------------------------------------------------------------------
# Save and validate JSON
# -----------------------------------------------------------------------------

output_path = AUDIT_DIR / "raw_file_audit.json"

with output_path.open("w", encoding="utf-8") as handle:
    json.dump(
        audit_records,
        handle,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

with output_path.open("r", encoding="utf-8") as handle:
    json.load(handle)


# -----------------------------------------------------------------------------
# Summary
# -----------------------------------------------------------------------------

print(f"Audit saved to: {project_relative_path(output_path)}")
print(f"Registry-managed raw files audited: {len(raw_file_audit):,}")
print("JSON validation passed.")

raw_file_audit

Manifest-managed datasets excluded from this audit:
  - tcga: 43,463 filesystem files

Audit saved to: data/interim/qc/raw_file_audit.json
Registry-managed raw files audited: 6
JSON validation passed.


,relative_path,file_name,dataset,source_database,provider,release,download_page_url,file_url,size_bytes,sha256
0,data/raw/depmap/Model.csv,Model.csv,depmap,DepMap,None,Public 24Q4,https://depmap.org/portal/download/,https://depmap.org/portal/data_page/?tab=allDa...,645696,b7a0c1385e6cef30132b56aff61f1261d11e3f490490b3...
1,data/raw/depmap/OmicsDefaultModelProfiles.csv,OmicsDefaultModelProfiles.csv,depmap,DepMap,None,Public 24Q4,https://depmap.org/portal/download/,https://depmap.org/portal/data_page/?tab=allDa...,90080,096a96c39d88374cb7c37058816e6cde29d6617b820b8a...
2,data/raw/depmap/OmicsExpressionProteinCodingGe...,OmicsExpressionProteinCodingGenesTPMLogp1.csv,depmap,DepMap,None,Public 24Q4,https://depmap.org/portal/download/,https://depmap.org/portal/data_page/?tab=allDa...,506628654,2a71dc94110efcc0221eae821bb93a9f03b54bea16f005...
3,data/raw/depmap/OmicsProfiles.csv,OmicsProfiles.csv,depmap,DepMap,None,Public 24Q4,https://depmap.org/portal/download/,https://depmap.org/portal/data_page/?tab=allDa...,254733,0bb19ee57935d4b213d378a4db9caacaa1db7318a63ae8...
4,data/raw/depmap/OmicsSomaticMutations.csv,OmicsSomaticMutations.csv,depmap,DepMap,None,Public 24Q4,https://depmap.org/portal/download/,https://depmap.org/portal/data_page/?tab=allDa...,338945382,92e86ab6a8457ef99908daf2a2c82d9529320e23cd6c6b...
5,data/raw/gdsc/GDSC2_fitted_dose_response_27Oct...,GDSC2_fitted_dose_response_27Oct23.xlsx,gdsc,Genomics of Drug Sensitivity in Cancer (GDSC),Cell Model Passports,GDSC2,https://cellmodelpassports.sanger.ac.uk/downloads,https://cmp.cog.sanger.ac.uk/download/GDSC2_fi...,22112537,0e99120a14a003dcbd3c27c020f1cfd35e38ee091067a2...
